# Modulo 1 · Lezione 11 — Programmazione Orientata agli Oggetti II

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vedremo oggi?

| # | Blocco | Cosa imparerai |
|---|--------|----------------|
| 1 | Ereditarietà | Riutilizzare ed estendere classi esistenti |
| 2 | `super()` | Chiamare il costruttore e i metodi del genitore |
| 3 | Override dei metodi | Cambiare il comportamento nella sottoclasse |
| 4 | Ereditarietà multipla e MRO | Ereditare da più di una classe |
| 5 | Polimorfismo | Stessa interfaccia, comportamenti diversi |
| 6 | Classi astratte | Definire contratti con `ABC` |
| 7 | Incapsulamento avanzato | `__slots__`, proprietà calcolate, pattern |
| 8 | Pratica | Sistema bancario |

> 💡 **Idea centrale:** l'ereditarietà ci permette di costruire su ciò che già esiste senza riscriverlo. Il polimorfismo ci permette di trattare oggetti diversi in modo uniforme. Insieme, rendono il codice estensibile: aggiungere un nuovo tipo di conto bancario non rompe nulla di ciò che già funziona.

---
## 1. Ereditarietà

L'ereditarietà permette a una classe (**sottoclasse** o classe figlia) di ereditare attributi e metodi da un'altra classe (**superclasse** o classe genitore), aggiungendo o modificando ciò che serve.

```
Classe genitore (superclasse)
     Animale
  nome, energia
  mangia(), dorme()
       │
       ├──────────────────┐
       ▼                  ▼
     Cane               Gatto
  + razza             + indoor
  + abbaia()          + fa_le_fusa()
  (eredita mangia,    (eredita mangia,
   dorme)              dorme)
```

**Sintassi:**
```python
class SottoClasse(SuperClasse):
    ...
```

**Regola d'oro:** usa l'ereditarietà quando la relazione è «**è un**». Un `Cane` **è un** `Animale` ✅. Un `Cane` **ha un** collare — quella è composizione, non ereditarietà.

In [ ]:
class Animale:
    def __init__(self, nome, energia=100):
        self.nome    = nome
        self.energia = energia

    def mangia(self, quantita=20):
        self.energia += quantita
        print(f"{self.nome} mangia e recupera {quantita} di energia (totale: {self.energia})")

    def dorme(self, ore=8):
        recuperata    = ore * 5
        self.energia += recuperata
        print(f"{self.nome} dorme {ore}h e recupera {recuperata} di energia (totale: {self.energia})")

    def __str__(self):
        return f"{type(self).__name__}({self.nome}, energia={self.energia})"


class Cane(Animale):          # Cane eredita da Animale
    def __init__(self, nome, razza):
        super().__init__(nome)  # chiama __init__ del genitore
        self.razza = razza

    def abbaia(self):
        self.energia -= 5
        print(f"{self.nome} abbaia: Bau! (energia: {self.energia})")


class Gatto(Animale):
    def __init__(self, nome, indoor=True):
        super().__init__(nome)
        self.indoor = indoor

    def fa_le_fusa(self):
        print(f"{self.nome} fa le fusa... 😸")


rex  = Cane("Rex",  "Pastore Tedesco")
misi = Gatto("Misi")

rex.mangia()                   # ereditato da Animale
rex.abbaia()                   # proprio di Cane
misi.dorme(ore=12)             # ereditato da Animale
misi.fa_le_fusa()              # proprio di Gatto

print()
print(rex)
print(misi)

# Controlli di tipo
print(f"\nrex  è Animale: {isinstance(rex,  Animale)}")   # True
print(f"misi è Cane:    {isinstance(misi, Cane)}")     # False
print(f"Cane sottoclasse di Animale: {issubclass(Cane, Animale)}")  # True

---
## 2. `super()` — Chiamare il genitore

`super()` restituisce un proxy verso l'oggetto genitore. Lo usiamo per:
1. Chiamare l'`__init__` del genitore senza ripeterne il codice.
2. Estendere un metodo del genitore invece di sostituirlo.

```python
class Figlio(Genitore):
    def __init__(self, a, b, c):
        super().__init__(a, b)   # inizializza ciò del genitore
        self.c = c               # aggiunge il proprio

    def metodo(self):
        super().metodo()         # esegue prima quello del genitore
        # ... e poi aggiunge comportamento extra
```

In [ ]:
class Veicolo:
    def __init__(self, marca, modello, velocita_max):
        self.marca         = marca
        self.modello       = modello
        self.velocita_max  = velocita_max
        self.velocita      = 0

    def accelera(self, incremento):
        self.velocita = min(self.velocita + incremento, self.velocita_max)
        print(f"{self.marca} {self.modello}: {self.velocita} km/h")

    def info(self):
        return f"{self.marca} {self.modello} (max {self.velocita_max} km/h)"


class Auto(Veicolo):
    def __init__(self, marca, modello, velocita_max, porte=4):
        super().__init__(marca, modello, velocita_max)  # delega al genitore
        self.porte = porte

    def info(self):
        base = super().info()     # ottiene l'info del genitore
        return f"{base}, {self.porte} porte"  # e la estende


class AutoElettrica(Auto):
    def __init__(self, marca, modello, velocita_max, batteria_kwh):
        super().__init__(marca, modello, velocita_max)  # chiama Auto.__init__
        self.batteria_kwh = batteria_kwh
        self.carica       = 100   # percentuale

    def accelera(self, incremento):
        super().accelera(incremento)   # comportamento base
        consumo      = incremento * 0.1
        self.carica  = max(0, self.carica - consumo)
        print(f"  Batteria: {self.carica:.1f}%")

    def info(self):
        base = super().info()
        return f"{base}, batteria {self.batteria_kwh} kWh"


auto  = Auto("Fiat", "Panda", 180, porte=5)
tesla = AutoElettrica("Tesla", "Model 3", 250, batteria_kwh=75)

print(auto.info())
auto.accelera(100)

print()
print(tesla.info())
tesla.accelera(80)
tesla.accelera(120)

# La gerarchia di ereditarietà
print(f"\nMRO di AutoElettrica:")
for cls in AutoElettrica.__mro__:
    print(f"  {cls.__name__}")

---
## 3. Override dei metodi

Una sottoclasse può **sovrascrivere** qualsiasi metodo del genitore per cambiarne il comportamento. Python esegue sempre il metodo più specifico (quello della sottoclasse).

```
oggetto.metodo()
      │
      └─► Python cerca in: SottoClasse → Genitore → Nonno → object
          Esegue il primo che trova
```

In [ ]:
class Figura:
    def area(self):
        raise NotImplementedError("Ogni figura deve implementare area()")

    def descrizione(self):
        return f"{type(self).__name__} con area {self.area():.2f}"


class Cerchio(Figura):
    def __init__(self, raggio):
        self.raggio = raggio

    def area(self):            # sovrascrive quello del genitore
        import math
        return math.pi * self.raggio ** 2


class Rettangolo(Figura):
    def __init__(self, larghezza, altezza):
        self.larghezza = larghezza
        self.altezza   = altezza

    def area(self):            # sovrascrive quello del genitore
        return self.larghezza * self.altezza


class Triangolo(Figura):
    def __init__(self, base, altezza):
        self.base    = base
        self.altezza = altezza

    def area(self):
        return (self.base * self.altezza) / 2


figure = [Cerchio(5), Rettangolo(6, 4), Triangolo(3, 8)]

for fig in figure:
    print(fig.descrizione())   # descrizione() è in Figura
                               # ma chiama area() di ogni sottoclasse

---
## 4. Ereditarietà multipla e MRO

Python permette di ereditare da più di una classe. Il **MRO** (Method Resolution Order) definisce l'ordine in cui Python cerca i metodi quando c'è ereditarietà multipla.

```python
class C(A, B):   # eredita da A e da B
    pass
```

Il MRO segue l'algoritmo **C3 linearization**: da sinistra a destra, dando priorità alla sottoclasse più specifica. Puoi consultarlo con `Classe.__mro__` o `Classe.mro()`.

> In pratica, l'ereditarietà multipla si usa principalmente per i **mixin** — classi piccole che aggiungono una funzionalità concreta senza costituire una gerarchia completa.

In [ ]:
# Pattern Mixin — classi che aggiungono funzionalità senza essere gerarchie

class LogMixin:
    """Aggiunge la capacità di logging a qualsiasi classe."""
    def log(self, messaggio):
        print(f"[LOG] {type(self).__name__}: {messaggio}")


class JsonMixin:
    """Aggiunge la serializzazione JSON a qualsiasi classe che abbia __dict__."""
    def to_json(self):
        import json
        dati = {k: v for k, v in self.__dict__.items() if not k.startswith("_")}
        return json.dumps(dati, ensure_ascii=False, indent=2)


class Prodotto:
    def __init__(self, nome, prezzo):
        self.nome   = nome
        self.prezzo = prezzo

    def applica_sconto(self, pct):
        self.prezzo = round(self.prezzo * (1 - pct / 100), 2)


class ProdottoRicco(LogMixin, JsonMixin, Prodotto):
    """Prodotto con logging e serializzazione JSON."""
    def __init__(self, nome, prezzo, categoria):
        super().__init__(nome, prezzo)
        self.categoria = categoria

    def applica_sconto(self, pct):
        self.log(f"Applico {pct}% di sconto su {self.prezzo} €")
        super().applica_sconto(pct)
        self.log(f"Prezzo finale: {self.prezzo} €")


p = ProdottoRicco("Laptop", 1200, "Elettronica")
p.applica_sconto(15)
print()
print(p.to_json())
print()
print("MRO:", [c.__name__ for c in ProdottoRicco.__mro__])

---
## 5. Polimorfismo

Il polimorfismo significa «molte forme». In POO: **la stessa chiamata a un metodo produce risultati diversi a seconda del tipo dell'oggetto**.

Non serve sapere quale tipo concreto è un oggetto per chiamare il suo metodo — basta che tutti condividano la stessa interfaccia.

```python
# Polimorfismo in azione
for figura in [Cerchio(5), Rettangolo(4,6), Triangolo(3,8)]:
    print(figura.area())   # ogni oggetto risponde a modo suo
```

Python ha anche il polimorfismo «duck typing»: se un oggetto ha il metodo di cui hai bisogno, non importa il suo tipo esatto (*«se cammina come un'anatra e starnazza come un'anatra, è un'anatra»*).

In [ ]:
# Polimorfismo tramite ereditarietà
class Notifica:
    def __init__(self, destinatario, messaggio):
        self.destinatario = destinatario
        self.messaggio    = messaggio

    def invia(self):
        raise NotImplementedError


class Email(Notifica):
    def invia(self):
        print(f"📧 Email  → {self.destinatario}: {self.messaggio}")


class SMS(Notifica):
    def invia(self):
        print(f"📱 SMS    → {self.destinatario}: {self.messaggio[:160]}")


class NotificaPush(Notifica):
    def invia(self):
        print(f"🔔 Push   → {self.destinatario}: {self.messaggio}")


def invia_tutte(notifiche):
    """Non sa né le interessa il tipo concreto — chiama solo .invia()."""
    for notif in notifiche:
        notif.invia()


coda = [
    Email("anna@mail.com",  "Il tuo ordine è stato spedito"),
    SMS("+39 333 000 0001",  "Codice di verifica: 4821"),
    NotificaPush("device_xyz", "Nuova offerta disponibile"),
    Email("luigi@mail.com", "Fattura allegata"),
]

invia_tutte(coda)

In [ ]:
# Duck typing — polimorfismo senza ereditarietà
class FilePDF:
    def leggi(self):
        return "Contenuto del PDF"

class FileCSV:
    def leggi(self):
        return "col1,col2\n1,2\n3,4"

class FileJSON:
    def leggi(self):
        return '{"chiave": "valore"}'

# Questa funzione funziona con qualsiasi oggetto che abbia .leggi()
# Non importa se ereditano da una classe comune o no
def elabora(file):
    contenuto = file.leggi()
    print(f"{type(file).__name__}: {len(contenuto)} caratteri")

for file in [FilePDF(), FileCSV(), FileJSON()]:
    elabora(file)

---
## 6. Classi astratte con `ABC`

Una **classe astratta** è una classe che non può essere istanziata direttamente — serve solo come modello per le sue sottoclassi. Definisce metodi che **devono** essere implementati da qualsiasi sottoclasse.

```python
from abc import ABC, abstractmethod

class MiaAstratta(ABC):
    @abstractmethod
    def metodo_obbligatorio(self):
        pass
```

Se una sottoclasse non implementa tutti i metodi astratti, Python lancia `TypeError` quando si prova a istanziarla.

**Quando usare ABC vs `raise NotImplementedError`?**

| | `raise NotImplementedError` | `ABC + @abstractmethod` |
|-|----------------------------|--------------------------|
| L'errore compare... | Alla chiamata del metodo | Al tentativo di istanziare |
| Si può istanziare la classe genitore | Sì | No |
| Garanzia di implementazione | A runtime | Alla costruzione |
| Quando usarlo | Script semplici | Contratti solidi nelle librerie |

In [ ]:
from abc import ABC, abstractmethod

class ContoBancario(ABC):
    """Contratto base per tutti i conti bancari."""

    def __init__(self, intestatario, saldo_iniziale=0):
        self.intestatario = intestatario
        self._saldo        = float(saldo_iniziale)

    @property
    def saldo(self):
        return self._saldo

    @abstractmethod
    def deposita(self, quantita):
        """Ogni tipo di conto definisce le proprie regole di deposito."""

    @abstractmethod
    def preleva(self, quantita):
        """Ogni tipo di conto definisce le proprie regole di prelievo."""

    @abstractmethod
    def tipo_conto(self):
        """Restituisce il nome del tipo di conto."""

    # Metodo concreto disponibile per tutte le sottoclassi
    def riepilogo(self):
        return f"[{self.tipo_conto()}] {self.intestatario} — Saldo: {self._saldo:,.2f} €"

    def __str__(self):
        return self.riepilogo()


# Provare a istanziare la classe astratta direttamente
try:
    conto = ContoBancario("Anna", 1000)
except TypeError as e:
    print(f"TypeError: {e}")

# Classe incompleta — non implementa tutti gli astratti
class ContoIncompleto(ContoBancario):
    def deposita(self, quantita):
        self._saldo += quantita
    # mancano preleva() e tipo_conto()

try:
    c = ContoIncompleto("Luigi", 500)
except TypeError as e:
    print(f"TypeError (incompleto): {e}")

---
## 7. Incapsulamento avanzato

Ampliamo i concetti di incapsulamento della lezione precedente con tecniche più precise.

In [ ]:
# __slots__ — dichiara gli attributi permessi e risparmia memoria
class PuntoNormale:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    # Ha __dict__ → si possono aggiungere nuovi attributi

class PuntoSlots:
    __slots__ = ("x", "y")   # solo questi attributi sono permessi

    def __init__(self, x, y):
        self.x = x
        self.y = y

p_normale = PuntoNormale(1, 2)
p_slots   = PuntoSlots(3, 4)

p_normale.z = 99   # permesso
print(f"Normale: x={p_normale.x}, z={p_normale.z}, dict={p_normale.__dict__}")

try:
    p_slots.z = 99  # proibito da __slots__
except AttributeError as e:
    print(f"Slots: {e}")

# Vantaggio di __slots__: meno memoria (niente __dict__)
import sys
print(f"\nDimensione PuntoNormale: {sys.getsizeof(p_normale.__dict__)} byte (solo il dict)")
print(f"PuntoSlots non ha __dict__: {'__dict__' not in dir(p_slots)}")

In [ ]:
# Proprietà calcolate e pattern di validazione a catena
class Temperatura:
    def __init__(self, celsius):
        self.celsius = celsius   # usa il setter

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, valore):
        if valore < -273.15:
            raise ValueError(f"{valore}°C è sotto lo zero assoluto")
        self._celsius = float(valore)

    @property
    def fahrenheit(self):               # solo getter — calcolato
        return self._celsius * 9/5 + 32

    @fahrenheit.setter
    def fahrenheit(self, valore):       # permette di assegnare in °F
        self.celsius = (valore - 32) * 5/9

    @property
    def kelvin(self):
        return self._celsius + 273.15

    def __str__(self):
        return f"{self._celsius:.1f}°C = {self.fahrenheit:.1f}°F = {self.kelvin:.2f}K"


t = Temperatura(100)
print(t)

t.fahrenheit = 32   # assegniamo in Fahrenheit
print(t)            # si converte automaticamente

try:
    t.celsius = -300
except ValueError as e:
    print(f"Errore: {e}")

---
## 8. Pratica — Sistema bancario

Costruiamo un sistema bancario che utilizza tutti i concetti di POO II: ereditarietà con `super()`, classi astratte con `ABC`, polimorfismo, incapsulamento e composizione.

---

### Preparazione — Architettura del sistema

```
ContoBancario (ABC)
  ├── deposita()  @abstractmethod
  ├── preleva()   @abstractmethod
  ├── tipo_conto() @abstractmethod
  └── riepilogo() [concreto]
        │
        ├── ContoCorrente
        │     senza limite di prelievo, commissione per scoperto
        │
        ├── ContoRisparmio
        │     non permette lo scoperto, genera interessi
        │
        └── ContoVincolato
              bloccato fino alla data di scadenza

Banca
  gestisce clienti e conti
  calcola statistiche globali
```

In [ ]:
from abc import ABC, abstractmethod
from datetime import date, datetime

# ─── Eccezioni del dominio bancario ────────────────────
class ErroreBancario(Exception):
    pass

class FondiInsufficienti(ErroreBancario):
    pass

class ContoBloccato(ErroreBancario):
    pass

class QuantitaNonValida(ErroreBancario):
    pass

# ─── Classe base astratta ─────────────────────────────────
class ContoBancario(ABC):
    _contatore = 0

    def __init__(self, intestatario, saldo_iniziale=0.0):
        ContoBancario._contatore += 1
        self.numero       = f"IT{ContoBancario._contatore:08d}"
        self.intestatario = intestatario
        self._saldo       = 0.0
        self._storico      = []
        if saldo_iniziale > 0:
            self._registra("Apertura", saldo_iniziale)
            self._saldo = float(saldo_iniziale)

    # ─── Proprietà saldo ─────────────────────────────────
    @property
    def saldo(self):
        return self._saldo

    # ─── Metodi astratti ──────────────────────────────
    @abstractmethod
    def deposita(self, quantita):
        pass

    @abstractmethod
    def preleva(self, quantita):
        pass

    @abstractmethod
    def tipo_conto(self):
        pass

    # ─── Metodi concreti (condivisi da tutte) ───────
    def _valida_quantita(self, quantita):
        if not isinstance(quantita, (int, float)) or quantita <= 0:
            raise QuantitaNonValida(f"Quantità non valida: {quantita}. Deve essere un numero positivo")

    def _registra(self, concetto, importo, saldo_risultato=None):
        self._storico.append({
            "data":     datetime.now().strftime("%Y-%m-%d %H:%M"),
            "concetto": concetto,
            "importo":  importo,
            "saldo":    saldo_risultato if saldo_risultato is not None else self._saldo,
        })

    def storico(self, ultimi=None):
        movs = self._storico[-ultimi:] if ultimi else self._storico
        print(f"\n{'─'*60}")
        print(f"  Storico: {self.numero} — {self.intestatario}")
        print(f"{'─'*60}")
        print(f"  {'Data':<17} {'Concetto':<20} {'Importo':>10} {'Saldo':>10}")
        print(f"{'─'*60}")
        for m in movs:
            segno = "+" if m["importo"] >= 0 else ""
            print(f"  {m['data']:<17} {m['concetto']:<20} {segno}{m['importo']:>9.2f} {m['saldo']:>10.2f}")
        print(f"{'─'*60}")

    def riepilogo(self):
        return f"[{self.tipo_conto()}] {self.numero} — {self.intestatario} | {self._saldo:,.2f} €"

    def __str__(self):
        return self.riepilogo()

    def __repr__(self):
        return f"{type(self).__name__}(numero={self.numero!r}, intestatario={self.intestatario!r}, saldo={self._saldo})"

    def __eq__(self, altro):
        if not isinstance(altro, ContoBancario):
            return NotImplemented
        return self.numero == altro.numero

    def __lt__(self, altro):
        return self._saldo < altro._saldo


print("✅ Classe base ContoBancario definita")

---
### Esercizio 1 — `ContoCorrente` e `ContoRisparmio`

**Consegna:** Implementa due sottoclassi di `ContoBancario`:

**`ContoCorrente`:**
- Permette lo scoperto fino a un limite (`limite_scoperto`, di default 0).
- Se va in scoperto, addebita una **commissione del 5%** sull'importo prelevato.
- `deposita` e `preleva` registrano il movimento in `_storico`.

**`ContoRisparmio`:**
- **Non permette** saldo negativo — lancia `FondiInsufficienti`.
- Ha un metodo `applica_interesse(tasso_annuo)` che accredita gli interessi e li registra.
- Il deposito minimo è 10 €.

**Esempio:**
```python
cc = ContoCorrente("Anna Rossi", saldo_iniziale=500, limite_scoperto=200)
cc.preleva(600)    # scoperto: addebita commissione
cr = ContoRisparmio("Luigi Bianchi", saldo_iniziale=1000)
cr.applica_interesse(2.5)
```

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
class ContoCorrente(ContoBancario):
    COMMISSIONE_SCOPERTO = 0.05   # 5%

    def __init__(self, intestatario, saldo_iniziale=0.0, limite_scoperto=0.0):
        super().__init__(intestatario, saldo_iniziale)
        self.limite_scoperto = float(limite_scoperto)

    def tipo_conto(self):
        return "Conto Corrente"

    def deposita(self, quantita):
        self._valida_quantita(quantita)
        self._saldo += quantita
        self._registra("Accredito", quantita, self._saldo)
        return self._saldo

    def preleva(self, quantita):
        self._valida_quantita(quantita)
        saldo_risultante = self._saldo - quantita

        if saldo_risultante < -self.limite_scoperto:
            raise FondiInsufficienti(
                f"Fondi insufficienti. Saldo: {self._saldo:.2f} €, "
                f"limite di scoperto: {self.limite_scoperto:.2f} €"
            )

        self._saldo -= quantita
        self._registra("Prelievo", -quantita, self._saldo)

        # Addebita commissione se entra in scoperto
        if self._saldo < 0:
            commissione = round(abs(self._saldo) * self.COMMISSIONE_SCOPERTO, 2)
            self._saldo -= commissione
            self._registra("Commissione scoperto", -commissione, self._saldo)
            print(f"⚠️  Scoperto. Commissione applicata: {commissione:.2f} €")

        return self._saldo


class ContoRisparmio(ContoBancario):
    DEPOSITO_MINIMO = 10.0

    def tipo_conto(self):
        return "Conto Risparmio"

    def deposita(self, quantita):
        self._valida_quantita(quantita)
        if quantita < self.DEPOSITO_MINIMO:
            raise QuantitaNonValida(
                f"Il deposito minimo è {self.DEPOSITO_MINIMO:.2f} €"
            )
        self._saldo += quantita
        self._registra("Accredito", quantita, self._saldo)
        return self._saldo

    def preleva(self, quantita):
        self._valida_quantita(quantita)
        if quantita > self._saldo:
            raise FondiInsufficienti(
                f"Fondi insufficienti. Saldo: {self._saldo:.2f} €, "
                f"prelievo richiesto: {quantita:.2f} €"
            )
        self._saldo -= quantita
        self._registra("Prelievo", -quantita, self._saldo)
        return self._saldo

    def applica_interesse(self, tasso_annuo):
        if tasso_annuo <= 0:
            raise QuantitaNonValida("Il tasso di interesse deve essere positivo")
        interesse = round(self._saldo * tasso_annuo / 100, 2)
        self._saldo += interesse
        self._registra(f"Interesse {tasso_annuo}%", interesse, self._saldo)
        print(f"💰 Interesse applicato: +{interesse:.2f} € (tasso {tasso_annuo}%)")
        return interesse


# ─── Prove ─────────────────────────────────────────────
ContoBancario._contatore = 0

cc = ContoCorrente("Anna Rossi", saldo_iniziale=500, limite_scoperto=200)
print(cc)

cc.deposita(300)
cc.preleva(400)
cc.preleva(500)   # entra in scoperto → commissione

try:
    cc.preleva(300)   # supera il limite
except FondiInsufficienti as e:
    print(f"❌ {e}")

cc.storico()

print()
cr = ContoRisparmio("Luigi Bianchi", saldo_iniziale=1000)
print(cr)
cr.deposita(500)
cr.applica_interesse(2.5)
cr.preleva(200)

try:
    cr.deposita(5)   # sotto il minimo
except QuantitaNonValida as e:
    print(f"❌ {e}")

cr.storico()
```

**Punti chiave:**
- `super().__init__()` in entrambe le sottoclassi delega l'inizializzazione comune (numero, storico, validazione del saldo iniziale) al genitore.
- `_valida_quantita` è nel genitore — le sottoclassi la chiamano senza duplicare la logica.
- La commissione di scoperto viene calcolata **dopo** aver aggiornato `_saldo` — così la percentuale si applica sull'importo scoperto reale.
- `COMMISSIONE_SCOPERTO` e `DEPOSITO_MINIMO` sono attributi di classe — facili da cambiare in un solo punto.
</details>

---
### Esercizio 2 — `ContoVincolato`

**Consegna:** Implementa `ContoVincolato(ContoBancario)`:

- Riceve `intestatario`, `saldo_iniziale`, `tasso_annuo` (es: `3.5`) e `mesi_vincolo` (es: `12`).
- Calcola e salva `data_scadenza` come `date`.
- **`preleva`** lancia `ContoBloccato` se non è arrivata la data di scadenza.
- **`deposita`** lancia `ContoBloccato` sempre (non ammette ulteriori depositi dopo l'apertura).
- **`liquida()`** — disponibile solo dopo la scadenza — calcola l'interesse accumulato, lo somma al saldo e restituisce il totale. Registra il movimento.
- **`giorni_rimanenti`** — proprietà che restituisce i giorni rimasti alla scadenza (0 se è già scaduto).
- **`riepilogo`** — sovrascrive per includere la data di scadenza e il tasso.

**Esempio:**
```python
cv = ContoVincolato("Marta Verdi", 10000, tasso_annuo=3.5, mesi_vincolo=12)
print(cv.giorni_rimanenti)   # ~365
cv.liquida()                 # ContoBloccato se non è ancora scaduto
```

In [ ]:
# ✏️ IL TUO CODICE QUI
from datetime import date, timedelta


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
from datetime import date, timedelta

class ContoVincolato(ContoBancario):

    def __init__(self, intestatario, saldo_iniziale, tasso_annuo, mesi_vincolo):
        super().__init__(intestatario, saldo_iniziale)
        self.tasso_annuo   = float(tasso_annuo)
        self.mesi_vincolo  = int(mesi_vincolo)
        self.data_apertura = date.today()
        self.data_scadenza = date.today() + timedelta(days=mesi_vincolo * 30)
        self._liquidato     = False

    def tipo_conto(self):
        return "Vincolato"

    @property
    def giorni_rimanenti(self):
        delta = (self.data_scadenza - date.today()).days
        return max(0, delta)

    @property
    def e_scaduto(self):
        return date.today() >= self.data_scadenza

    def deposita(self, quantita):
        raise ContoBloccato(
            "I depositi aggiuntivi non sono permessi in un conto vincolato"
        )

    def preleva(self, quantita):
        if not self.e_scaduto:
            raise ContoBloccato(
                f"Il conto scade il {self.data_scadenza}. "
                f"Mancano {self.giorni_rimanenti} giorni."
            )
        self._valida_quantita(quantita)
        if quantita > self._saldo:
            raise FondiInsufficienti(f"Saldo disponibile: {self._saldo:.2f} €")
        self._saldo -= quantita
        self._registra("Prelievo", -quantita, self._saldo)
        return self._saldo

    def liquida(self):
        if not self.e_scaduto:
            raise ContoBloccato(
                f"Non si può liquidare prima del {self.data_scadenza} "
                f"({self.giorni_rimanenti} giorni rimanenti)"
            )
        if self._liquidato:
            raise ErroreBancario("Questo conto è già stato liquidato")
        # Interesse semplice proporzionale al vincolo
        frazione_anno = self.mesi_vincolo / 12
        interesse     = round(self._saldo * (self.tasso_annuo / 100) * frazione_anno, 2)
        self._saldo  += interesse
        self._registra(f"Liquidazione {self.tasso_annuo}% / {self.mesi_vincolo}m", interesse, self._saldo)
        self._liquidato = True
        print(f"✅ Vincolato liquidato. Interesse: +{interesse:.2f} € → Saldo totale: {self._saldo:,.2f} €")
        return self._saldo

    def riepilogo(self):
        stato = "SCADUTO" if self.e_scaduto else f"{self.giorni_rimanenti}g rimanenti"
        return (
            f"[{self.tipo_conto()}] {self.numero} — {self.intestatario} | "
            f"{self._saldo:,.2f} € | {self.tasso_annuo}% / {self.mesi_vincolo}m | {stato}"
        )


# ─── Prove ─────────────────────────────────────────────
cv = ContoVincolato("Marta Verdi", 10_000, tasso_annuo=3.5, mesi_vincolo=12)
print(cv)
print(f"Giorni rimanenti: {cv.giorni_rimanenti}")

try:
    cv.deposita(500)
except ContoBloccato as e:
    print(f"❌ Deposito: {e}")

try:
    cv.preleva(1000)
except ContoBloccato as e:
    print(f"❌ Prelievo: {e}")

try:
    cv.liquida()
except ContoBloccato as e:
    print(f"❌ Liquidare: {e}")

# Simulare la scadenza (forzare la data per la prova)
cv.data_scadenza = date.today() - timedelta(days=1)
print(f"\nSimulazione scadenza...")
print(f"È scaduto? {cv.e_scaduto}")
cv.liquida()
cv.storico()
```

**Punti chiave:**
- `e_scaduto` come `@property` incapsula il confronto tra date — chi usa la classe non ha bisogno di sapere come viene calcolato.
- `_liquidato` evita che l'interesse venga applicato due volte se qualcuno chiama `liquida()` in un ciclo.
- `deposita` sovrascrive completamente il genitore per lanciare **sempre** un'eccezione — il conto vincolato non ammette ulteriori depositi.
- Forzare la data nelle prove (`cv.data_scadenza = ...`) è una tecnica standard per testare la logica temporale senza aspettare giorni reali.
</details>

---
### Esercizio 3 — Polimorfismo: operazioni su qualsiasi conto

**Consegna:** Scrivi funzioni che lavorino in modo **polimorfico** su qualsiasi tipo di conto:

1. `calcola_patrimonio(conti)` → somma il saldo di tutti i conti.
2. `conto_piu_ricco(conti)` → restituisce il conto con il saldo più alto.
3. `riepilogo_portafoglio(conti)` → stampa un report raggruppato per tipo di conto, con totali per tipo e totale globale.
4. `prova_prelievo(conto, quantita)` → prova a prelevare, cattura le eccezioni bancarie e restituisce `True/False`.

Queste funzioni **non sanno né importa loro** se il conto è corrente, risparmio o vincolato — chiamano solo l'interfaccia comune.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
from collections import defaultdict

def calcola_patrimonio(conti):
    """Somma il saldo di tutti i conti — polimorfismo puro."""
    return sum(c.saldo for c in conti)

def conto_piu_ricco(conti):
    """Restituisce il conto con il saldo più alto — usa __lt__ definito in ContoBancario."""
    if not conti:
        return None
    return max(conti)   # max() usa __lt__ → __gt__ internamente

def riepilogo_portafoglio(conti):
    """Report raggruppato per tipo — usa solo l'interfaccia di ContoBancario."""
    if not conti:
        print("Il portafoglio è vuoto.")
        return

    gruppi = defaultdict(list)
    for c in conti:
        gruppi[c.tipo_conto()].append(c)   # tipo_conto() è polimorfico

    totale_globale = calcola_patrimonio(conti)

    print(f"\n{'═'*62}")
    print(f"  RIEPILOGO PORTAFOGLIO — {len(conti)} conto/i")
    print(f"{'═'*62}")

    for tipo, gruppo in sorted(gruppi.items()):
        totale_tipo = sum(c.saldo for c in gruppo)
        print(f"\n  {tipo.upper()} ({len(gruppo)}) — Totale: {totale_tipo:,.2f} €")
        print(f"  {'─'*58}")
        for c in sorted(gruppo, reverse=True):   # ordinati per saldo desc
            print(f"  {c.riepilogo()}")           # riepilogo() è polimorfico

    print(f"\n{'═'*62}")
    print(f"  PATRIMONIO TOTALE: {totale_globale:,.2f} €")
    print(f"{'═'*62}")

def prova_prelievo(conto, quantita):
    """Prova a prelevare. Restituisce True se riesce, False altrimenti."""
    try:
        conto.preleva(quantita)
        print(f"✅ Prelievo di {quantita:.2f} € da {conto.numero} — nuovo saldo: {conto.saldo:.2f} €")
        return True
    except ContoBloccato as e:
        print(f"🔒 Conto bloccato ({conto.numero}): {e}")
        return False
    except FondiInsufficienti as e:
        print(f"💸 Fondi insufficienti ({conto.numero}): {e}")
        return False
    except QuantitaNonValida as e:
        print(f"⚠️  Quantità non valida: {e}")
        return False


# ─── Prove ─────────────────────────────────────────────
ContoBancario._contatore = 0

conti = [
    ContoCorrente("Anna Rossi",   saldo_iniziale=2500,  limite_scoperto=500),
    ContoCorrente("Luigi Bianchi", saldo_iniziale=800),
    ContoRisparmio("Marta Verdi",  saldo_iniziale=5000),
    ContoRisparmio("Carlo Neri",   saldo_iniziale=1200),
    ContoVincolato("Eva Martini",  10_000, tasso_annuo=3.5, mesi_vincolo=12),
]

print(f"Patrimonio totale: {calcola_patrimonio(conti):,.2f} €")
print(f"Conto più ricco:   {conto_piu_ricco(conti)}")

riepilogo_portafoglio(conti)

print("\n── Tentativi di prelievo ──")
prova_prelievo(conti[0], 1000)    # OK
prova_prelievo(conti[1], 5000)    # fondi insufficienti
prova_prelievo(conti[4], 1000)    # vincolato bloccato
```

**Punti chiave:**
- `c.tipo_conto()`, `c.riepilogo()`, `c.preleva()` — ogni chiamata esegue il codice della sottoclasse corretta. Le funzioni non usano `isinstance()` né `if type(c) == ...` — ciò romperebbe il polimorfismo.
- `max(conti)` funziona perché `ContoBancario` definisce `__lt__` — Python può confrontare qualsiasi coppia di conti senza conoscerne il tipo.
- `prova_prelievo` cattura `ErroreBancario` (base) in sottoclassi specifiche — l'ordine degli `except` conta: il più specifico per primo.
- `defaultdict(list)` raggruppa i conti per tipo senza dover controllare se la chiave esiste.
</details>

---
### Esercizio 4 — La classe `Banca`

**Consegna:** Crea la classe `Banca` che gestisce clienti e conti:

- **`apri_conto(tipo, intestatario, **kwargs)`** — crea un conto del tipo indicato (`"corrente"`, `"risparmio"`, `"vincolato"`), lo salva e lo restituisce.
- **`cerca_conto(numero)`** — restituisce il conto o lancia `KeyError`.
- **`trasferisci(numero_origine, numero_destinazione, quantita)`** — sposta denaro tra conti. Usa `preleva` e `deposita` — il polimorfismo fa il resto.
- **`totale_depositi()`** — somma di tutti i saldi positivi.
- **`report()`** — chiama `riepilogo_portafoglio` dell'esercizio precedente.
- **`__len__`** — numero di conti.
- **`__contains__`** — permette `numero in banca`.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
class Banca:
    TIPI = {
        "corrente":  ContoCorrente,
        "risparmio": ContoRisparmio,
        "vincolato": ContoVincolato,
    }

    def __init__(self, nome):
        self.nome   = nome
        self._conti = {}   # {numero: ContoBancario}

    def apri_conto(self, tipo, intestatario, **kwargs):
        if tipo not in self.TIPI:
            raise ValueError(f"Tipo di conto sconosciuto: '{tipo}'. Opzioni: {list(self.TIPI)}")
        ClasseConto = self.TIPI[tipo]
        conto = ClasseConto(intestatario, **kwargs)
        self._conti[conto.numero] = conto
        print(f"✅ Conto aperto: {conto}")
        return conto

    def cerca_conto(self, numero):
        conto = self._conti.get(numero)
        if conto is None:
            raise KeyError(f"Non esiste nessun conto con numero '{numero}'")
        return conto

    def trasferisci(self, numero_origine, numero_destinazione, quantita):
        origine      = self.cerca_conto(numero_origine)
        destinazione = self.cerca_conto(numero_destinazione)
        # Il polimorfismo gestisce le regole di ogni tipo di conto
        origine.preleva(quantita)
        destinazione.deposita(quantita)
        print(
            f"🔄 Trasferimento: {quantita:.2f} € "
            f"da {origine.numero} ({origine.intestatario}) "
            f"a {destinazione.numero} ({destinazione.intestatario})"
        )

    def totale_depositi(self):
        return sum(c.saldo for c in self._conti.values())

    def report(self):
        print(f"\n  Banca: {self.nome}")
        riepilogo_portafoglio(list(self._conti.values()))

    def __len__(self):
        return len(self._conti)

    def __contains__(self, numero):
        return numero in self._conti

    def __str__(self):
        return f"{self.nome} — {len(self)} conto/i, {self.totale_depositi():,.2f} € in depositi"


# ─── Prove ─────────────────────────────────────────────
ContoBancario._contatore = 0

banca = Banca("Banca Digitale Lombardia")

c1 = banca.apri_conto("corrente",  "Anna Rossi",   saldo_iniziale=3000, limite_scoperto=500)
c2 = banca.apri_conto("risparmio", "Luigi Bianchi", saldo_iniziale=8000)
c3 = banca.apri_conto("corrente",  "Marta Verdi",   saldo_iniziale=1500)
c4 = banca.apri_conto("vincolato", "Carlo Neri",    saldo_iniziale=20_000, tasso_annuo=4.0, mesi_vincolo=24)

print(f"\nConti nella banca: {len(banca)}")
print(f"{c1.numero} in banca: {c1.numero in banca}")

print("\n── Trasferimento ──")
try:
    banca.trasferisci(c1.numero, c2.numero, 1000)
except ErroreBancario as e:
    print(f"❌ {e}")

# Trasferire a un conto vincolato → ContoBloccato
print()
try:
    banca.trasferisci(c1.numero, c4.numero, 500)
except ContoBloccato as e:
    print(f"❌ Non si può trasferire a un conto vincolato: {e}")

banca.report()
```

**Punti chiave:**
- `TIPI = {"corrente": ContoCorrente, ...}` è il pattern **factory** — il dizionario mappa una stringa a una classe. `ClasseConto(intestatario, **kwargs)` crea l'istanza senza un lungo `if/elif`.
- `trasferisci` non fa nessun `isinstance` né `if tipo == ...` — chiama `preleva` e `deposita`, e il polimorfismo fa il resto. Se il conto vincolato rifiuta il deposito, l'eccezione si propaga da sola.
- `cerca_conto` usa `.get()` e lancia `KeyError` — stesso comportamento di un dizionario nativo, coerente con le aspettative di chi usa la classe.
</details>

---
### Esercizio 5 (sfida) — `ContoGiovani` ed ereditarietà a catena

**Consegna:** Crea `ContoGiovani(ContoRisparmio)` che eredita da `ContoRisparmio` e aggiunge:

- Si può aprire solo per intestatari con età tra 14 e 25 anni (riceve `eta` nel costruttore).
- Il deposito minimo scende a 5 € (sovrascrive la costante della classe genitore).
- `preleva` applica un **limite mensile di 500 €** — accumula i prelievi del mese corrente e lancia `ErroreBancario` se viene superato.
- `applica_interesse` applica un tasso del **50% superiore** rispetto al risparmio normale (bonus giovani).
- `compie_anni()` — se l'intestatario compie 26 anni, converte il conto in `ContoRisparmio` normale (restituisce una nuova istanza con lo stesso saldo e storico).

Integra `ContoGiovani` nel sistema di `Banca` aggiungendo `"giovani"` al dizionario `TIPI`.

Costruisci uno scenario completo e dimostra la catena di ereditarietà:
```
ContoBancario (ABC)
    └── ContoRisparmio
            └── ContoGiovani
```

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
from datetime import date

class ContoGiovani(ContoRisparmio):
    DEPOSITO_MINIMO  = 5.0    # sovrascrive quello di ContoRisparmio (10.0)
    LIMITE_MENSILE   = 500.0
    ETA_MINIMA       = 14
    ETA_MASSIMA      = 25
    BONUS_INTERESSE  = 1.5    # 50% in più rispetto al tasso base

    def __init__(self, intestatario, eta, saldo_iniziale=0.0):
        if not (self.ETA_MINIMA <= eta <= self.ETA_MASSIMA):
            raise ValueError(
                f"Il conto giovani è per persone da {self.ETA_MINIMA} "
                f"a {self.ETA_MASSIMA} anni. Età ricevuta: {eta}"
            )
        super().__init__(intestatario, saldo_iniziale)
        self.eta            = eta
        self._prelevato_mese = 0.0
        self._mese_attuale   = date.today().month

    def tipo_conto(self):
        return "Conto Giovani"

    def _resetta_limite_se_nuovo_mese(self):
        mese_oggi = date.today().month
        if mese_oggi != self._mese_attuale:
            self._prelevato_mese = 0.0
            self._mese_attuale    = mese_oggi

    def preleva(self, quantita):
        self._resetta_limite_se_nuovo_mese()
        if self._prelevato_mese + quantita > self.LIMITE_MENSILE:
            disponibile = self.LIMITE_MENSILE - self._prelevato_mese
            raise ErroreBancario(
                f"Limite mensile di {self.LIMITE_MENSILE:.2f} € raggiunto. "
                f"Hai già prelevato {self._prelevato_mese:.2f} €, "
                f"ti restano {disponibile:.2f} € disponibili questo mese."
            )
        super().preleva(quantita)   # delega la logica dei fondi al genitore
        self._prelevato_mese += quantita

    def applica_interesse(self, tasso_annuo):
        tasso_con_bonus = round(tasso_annuo * self.BONUS_INTERESSE, 4)
        print(f"🎁 Bonus giovani: tasso {tasso_annuo}% → {tasso_con_bonus}%")
        return super().applica_interesse(tasso_con_bonus)

    def compie_anni(self):
        self.eta += 1
        if self.eta > self.ETA_MASSIMA:
            print(f"🎂 {self.intestatario} ha compiuto {self.eta} anni. "
                  f"Il conto giovani si converte in conto risparmio standard.")
            nuovo = ContoRisparmio(self.intestatario, saldo_iniziale=0.0)
            nuovo._saldo   = self._saldo          # trasferire il saldo
            nuovo._storico = self._storico[:]      # copiare lo storico
            nuovo._registra("Conversione da Conto Giovani", 0, nuovo._saldo)
            return nuovo
        print(f"🎂 {self.intestatario} compie {self.eta} anni. Conto giovani attivo.")
        return self

    def riepilogo(self):
        prelevato = self._prelevato_mese
        base      = super().riepilogo()
        return f"{base} | Età: {self.eta} | Prelevato questo mese: {prelevato:.2f}/{self.LIMITE_MENSILE:.0f} €"


# Integrare in Banca
Banca.TIPI["giovani"] = ContoGiovani


# ─── Scenario completo ───────────────────────────────────
ContoBancario._contatore = 0
banca = Banca("Banca Digitale Lombardia")

cg = banca.apri_conto("giovani", "Sofia Marino", eta=20, saldo_iniziale=800)

print("\n── Operazioni conto giovani ──")
cg.deposita(50)          # minimo 5 € (non 10 come in ContoRisparmio)
cg.preleva(200)
cg.preleva(250)

try:
    cg.preleva(100)       # supera il limite mensile di 500 €
except ErroreBancario as e:
    print(f"❌ {e}")

cg.applica_interesse(2.0)   # riceverà 3.0% grazie al bonus
print(cg)

# Verificare la catena di ereditarietà
print("\n── Catena di ereditarietà ──")
print(f"MRO: {[c.__name__ for c in ContoGiovani.__mro__]}")
print(f"isinstance(cg, ContoRisparmio):  {isinstance(cg, ContoRisparmio)}")
print(f"isinstance(cg, ContoBancario):   {isinstance(cg, ContoBancario)}")

# Compiere anni e convertire
print("\n── Compiere anni ──")
for _ in range(6):                  # da 20 a 26
    cg = cg.compie_anni()

print(f"Tipo dopo la conversione: {type(cg).__name__}")
cg.storico(ultimi=5)
```

**Punti chiave:**
- `DEPOSITO_MINIMO = 5.0` in `ContoGiovani` sovrascrive l'attributo di classe del genitore senza toccare il genitore — l'ereditarietà degli attributi di classe funziona come quella dei metodi.
- `super().preleva(quantita)` in `ContoGiovani.preleva` delega la logica del saldo negativo a `ContoRisparmio` — non duplichiamo quella validazione.
- `compie_anni` restituisce `self` o una nuova istanza di `ContoRisparmio` — il codice chiamante salva solo il risultato (`cg = cg.compie_anni()`) e non gli importa quale dei due riceve.
- `Banca.TIPI["giovani"] = ContoGiovani` estende il sistema senza modificare la classe `Banca` — aperto all'estensione, chiuso alla modifica (principio Open/Closed).
</details>

---
## Riepilogo della lezione

### Ereditarietà

```python
class Figlio(Genitore):               # eredita tutto dal genitore
    def __init__(self, a, b, c):
        super().__init__(a, b)        # inizializza ciò del genitore
        self.c = c                    # aggiunge il proprio

    def metodo(self):                 # sovrascrive quello del genitore
        risultato = super().metodo()  # può chiamare prima il genitore
        return risultato + " extra"
```

### Classi astratte

```python
from abc import ABC, abstractmethod

class Base(ABC):
    @abstractmethod
    def metodo_obbligatorio(self):   # le sottoclassi DEVONO implementarlo
        pass

    def metodo_concreto(self):       # disponibile per tutte le sottoclassi
        return self.metodo_obbligatorio()  # può chiamare gli astratti
```

### Tabella di riferimento: relazioni tra classi

| Relazione | Quando | Esempio |
|----------|--------|---------|
| **Ereditarietà** (`è un`) | La sottoclasse è una versione specializzata | `ContoGiovani` è un `ContoRisparmio` |
| **Composizione** (`ha un`) | Una classe usa oggetti di un'altra | `Banca` ha dei `ContoBancario` |
| **Mixin** | Aggiungere funzionalità trasversale | `LogMixin`, `JsonMixin` |

### Polimorfismo — segnali di buon design

✅ Il codice chiamante **non usa `isinstance()`** per decidere cosa fare
✅ Aggiungere un nuovo tipo **non richiede di modificare** il codice esistente
✅ L'interfaccia comune (metodi astratti) **garantisce il contratto**

---

## Cosa vedremo nella prossima lezione?

Nella **Lezione 12** chiudiamo il Modulo 1 con un **progetto integratore** che userà tutto ciò che abbiamo imparato: strutture dati, funzioni, file, eccezioni e POO. Costruiremo un sistema completo dall'inizio alla fine.